**Homework, not run on the day.**

Same Gu–Kelly–Xiu design as `01a`, with named Fama–French characteristics. Do this after class.


# Gu–Kelly–Xiu pipeline — Fama–French characteristic version

**Paper:** Gu, S., Kelly, B., & Xiu, D. (2020). *Empirical Asset Pricing via Machine Learning.* *Review of Financial Studies*, 33(5), 2223–2273.

Still that research design — not a Fama–French *factor* replication and not CRSP/Compustat.

Sibling of `01a_gu_kelly_xiu_simplified.ipynb`. Same temporal split, same OOS \(R^2\), same long–short. The difference is the **feature story**:

| This notebook | Notebook 01a |
|---------------|--------------|
| Named characteristics (`size`, `bm`, `op`, `inv`, `mom`) + a small zoo | Anonymous `x0`–`x11` |
| First rungs = **FF3 / OLS-3 (GKX) / FF5** | First rung = OLS on all `x` |

This is a **characteristic model** (cross-section). It is **not** a time-series CAPM.

**Companion:** `sessions/01b_gu_fama_french.md` · slides `slides/01b_fama_french.md`


## 0. Setup

The unregularised OLS baseline overflows on collinear characteristics; the warnings are suppressed on purpose and the penalised models below are the point of this notebook.

In [ ]:
import warnings, numpy as np
warnings.filterwarnings("ignore", category=RuntimeWarning)
np.seterr(over="ignore", divide="ignore", invalid="ignore")
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(2020)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

## 1. Synthetic panel with Fama–French names

Ranked characteristics (paper spirit: cross-sectional rank transform):

- `size` — market equity (SMB)
- `bm` — book-to-market (HML)
- `op` — operating profitability (RMW)
- `inv` — investment (CMA)
- `mom` — momentum (Carhart; not in FF5)
- `z0`–`z6` — extra “zoo” characteristics

Planted expected return (unknown to the models):

$$
\mathbb{E}[r_{i,t+1}\mid x_{i,t}]
= \underbrace{\beta_s\,\mathrm{size}+\beta_b\,\mathrm{bm}}_{\text{linear FF3}}
+ \beta_o\,\mathrm{op}
+ \beta_{o2}\,\mathrm{op}^2
+ \beta_{sm}\,(\mathrm{size}\times\mathrm{mom})
$$

plus noise. FF OLS can get the linear block; leftover nonlinearity is why trees/nets can still win **on this DGP**.


In [ ]:
N, T = 200, 180  # stocks, months
try:
    dates = pd.date_range("2005-01-31", periods=T, freq="ME")
except ValueError:
    dates = pd.date_range("2005-01-31", periods=T, freq="M")

ff3 = ["size", "bm"]
ols3_gkx = ["size", "bm", "mom"]
ff5 = ["size", "bm", "op", "inv"]
named = ["size", "bm", "op", "inv", "mom"]
zoo = [f"z{j}" for j in range(7)]
features = named + zoo

rows = []
for dt in dates:
    X = rng.normal(size=(N, len(features)))
    X_rank = (pd.DataFrame(X).rank(axis=0, pct=True).to_numpy() - 0.5) * 2
    df_t = pd.DataFrame(X_rank, columns=features)
    size, bm, op, mom = df_t["size"], df_t["bm"], df_t["op"], df_t["mom"]
    f = (
        0.016 * size
        + 0.012 * bm
        + 0.010 * op
        + 0.015 * (op ** 2)
        + 0.022 * (size * mom)
    )
    df_t["stock"] = np.arange(N)
    df_t["date"] = dt
    df_t["f_true"] = f
    df_t["ret"] = f + rng.normal(scale=0.05, size=N)
    rows.append(df_t)

panel = pd.concat(rows, ignore_index=True)
print(panel.shape)
print("fundamentals FF3:", ff3)
print("OLS-3 (GKX):", ols3_gkx)
print("fundamentals FF5:", ff5)
print("full feature set:", features)
panel.head()

## 2. Temporal train / validation / test

**Frozen with notebook 01a:** same calendar cuts. Never shuffle stock–months.


In [ ]:
train_end = pd.Timestamp("2014-12-31")
val_end = pd.Timestamp("2016-12-31")

train = panel[panel["date"] <= train_end]
val = panel[(panel["date"] > train_end) & (panel["date"] <= val_end)]
test = panel[panel["date"] > val_end]

for name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s}: {df['date'].nunique():3d} months | {len(df):6d} rows")

y_train = train["ret"].to_numpy()
y_val = val["ret"].to_numpy()
y_test = test["ret"].to_numpy()

## 3. OOS \(R^2\) (predictive)

Same definition as notebook 01a (Gu–Kelly–Xiu spirit): benchmark forecast of 0.


In [ ]:
def r2_oos(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return 1.0 - np.sum((y_true - y_pred) ** 2) / np.sum(y_true ** 2)

## 4. Model ladder — FF OLS first, then Gu on the zoo

| Model | Features | Intent |
|-------|----------|--------|
| **FF3_OLS** | size, bm | Characteristic FF3 |
| **FF5_OLS** | size, bm, op, inv | FF 2015 (+ profitability, investment) |
| **OLS3_GKX** | size, bm, mom | GKX’s sparse OLS-3 baseline |
| OLS_all | named + zoo | Kitchen-sink linear (Gu’s OLS rung) |
| Ridge | named + zoo | Shrinkage |
| RF | named + zoo | Nonlinearities / interactions |
| Small MLP | named + zoo | Flexible approximation |

Hyperparameters: light touch on **validation** only.


In [ ]:
def fit_on(cols, model):
    model.fit(train[cols], y_train)
    return cols, model


ladder = [
    ("FF3_OLS", ff3, LinearRegression()),
    ("FF5_OLS", ff5, LinearRegression()),
    ("OLS3_GKX", ols3_gkx, LinearRegression()),
    ("OLS_all", features, LinearRegression()),
    ("Ridge", features, Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0)),
    ])),
    ("RF", features, RandomForestRegressor(
        n_estimators=200, max_depth=4, min_samples_leaf=50,
        random_state=2020, n_jobs=-1,
    )),
    ("MLP", features, Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPRegressor(
            hidden_layer_sizes=(32, 16), activation="relu",
            alpha=1e-3, max_iter=500, random_state=2020,
        )),
    ])),
]

fitted = {}
val_scores = []
for name, cols, model in ladder:
    cols, model = fit_on(cols, model)
    fitted[name] = (cols, model)
    pred = model.predict(val[cols])
    s = r2_oos(y_val, pred)
    val_scores.append({"model": name, "n_features": len(cols), "r2_oos_val": s})
    print(f"{name:8s}  p={len(cols):2d}  val R2_oos = {s:.4f}")

val_leaderboard = pd.DataFrame(val_scores).sort_values("r2_oos_val", ascending=False)
val_leaderboard.round(4)

## 5. Test-set comparison + long–short

Each month: sort on \(\hat r\), long top decile / short bottom. **ORACLE_f** uses the planted \(f\) (upper bound on this DGP, not in the paper).


In [ ]:
def long_short_returns(df, pred, q=0.1):
    out = df[["date", "ret"]].copy()
    out["pred"] = pred
    port = []
    for dt, g in out.groupby("date"):
        lo = g["pred"].quantile(q)
        hi = g["pred"].quantile(1 - q)
        long = g.loc[g["pred"] >= hi, "ret"].mean()
        short = g.loc[g["pred"] <= lo, "ret"].mean()
        port.append({"date": dt, "ls": long - short})
    return pd.DataFrame(port).set_index("date").sort_index()


def sharpe(monthly_returns):
    mu = monthly_returns.mean()
    sd = monthly_returns.std(ddof=1)
    return np.sqrt(12) * mu / sd if sd > 0 else np.nan


rows = []
ls_paths = {}
for name, (cols, model) in fitted.items():
    pred = model.predict(test[cols])
    ls = long_short_returns(test, pred)
    ls_paths[name] = ls["ls"]
    rows.append({
        "model": name,
        "r2_oos_test": r2_oos(y_test, pred),
        "ls_mean_monthly": ls["ls"].mean(),
        "ls_sharpe_ann": sharpe(ls["ls"]),
    })

oracle_pred = test["f_true"].to_numpy()
ls_oracle = long_short_returns(test, oracle_pred)
rows.append({
    "model": "ORACLE_f",
    "r2_oos_test": r2_oos(y_test, oracle_pred),
    "ls_mean_monthly": ls_oracle["ls"].mean(),
    "ls_sharpe_ann": sharpe(ls_oracle["ls"]),
})
ls_paths["ORACLE_f"] = ls_oracle["ls"]

results = pd.DataFrame(rows).sort_values("r2_oos_test", ascending=False)
results.round(4)

In [ ]:
cum = pd.DataFrame({k: (1 + v).cumprod() for k, v in ls_paths.items()})
ax = cum.plot(title="Cumulative long–short (test), q=0.1, zero costs")
ax.set_ylabel("growth of $1")
ax.legend(ncols=2, fontsize=8)
plt.tight_layout()

## 6. Discussion questions

1. OLS-3 (with mom) vs FF3 (without): how big is the hole that momentum fills?
2. Why can RF/MLP beat FF OLS **here** (`op²`, `size × mom`) — and why that need not be true in CRSP?
3. Characteristic OLS vs time-series CAPM: what object is \(\hat r_{i,t+1}\)?
4. Map this table back to Gu–Kelly–Xiu Table 1: which rows are “ours”?


## 7. Experiments — “do differently”

Keep test honest (tune only on val).


In [ ]:
# A) OLS-3 (with mom) vs FF3 (without) vs FF5
linear_sets = {
    "FF3": ff3,
    "OLS-3 (with mom)": ols3_gkx,
    "FF5": ff5,
}
lin_rows = []
for lab, cols in linear_sets.items():
    m = LinearRegression().fit(train[cols], y_train)
    p = m.predict(test[cols])
    ls = long_short_returns(test, p)
    lin_rows.append({
        "spec": lab,
        "n_features": len(cols),
        "r2_oos_test": r2_oos(y_test, p),
        "ls_sharpe_ann": sharpe(ls["ls"]),
    })
print(pd.DataFrame(lin_rows).round(4).to_string(index=False))

# B) Best-on-val model, wider long–short (q=0.2)
q_alt = 0.2
name = val_leaderboard.iloc[0]["model"]
cols, model = fitted[name]
ls_alt = long_short_returns(test, model.predict(test[cols]), q=q_alt)
print(f"\nBest-on-val model={name}, q={q_alt}")
print("mean monthly LS:", round(ls_alt["ls"].mean(), 4))
print("ann. Sharpe:    ", round(sharpe(ls_alt["ls"]), 4))

## Takeaway

The Fama–French OLS is the **named linear baseline**. Gu–Kelly–Xiu is the same OOS protocol with a larger characteristic set and a model ladder — not a claim that neural nets replace `size` and `bm`.

**Sibling:** `notebooks/01a_gu_kelly_xiu_simplified.ipynb` if you want anonymous \(x_j\) first.
